# Air Quality Category Prediction
Multi-class classification of air quality categories from pollutant measurements.

## Project Overview
Predict air quality category (Good/Moderate/Unhealthy/etc.) from pollutant concentration measurements across global cities.

## Learning Objectives
- Multi-class environmental classification
- Handle mixed numeric/categorical features
- Understand AQI categories and thresholds

## Problem Statement
Given pollutant measurements (PM2.5, PM10, NO2, SO2, CO, O3), predict the AQI category for a location.

## Why This Project Matters
Air quality monitoring protects public health. ML models can fill gaps where monitoring stations are sparse.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Kaggle: adityaramachandran27/world-air-quality-index-by-city-and-coordinates |
| **Target** | AQI Category (multi-class) |
| **Features** | Pollutant levels, city, coordinates |

## Dataset Source & License
Kaggle World Air Quality Index dataset. Check Kaggle for license.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
MAX_ROWS = 50000

## Dataset Loading

In [4]:
import kagglehub
path = kagglehub.dataset_download('adityaramachandran27/world-air-quality-index-by-city-and-coordinates')
import glob
csvs = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
print('Files:', csvs)
df = pd.read_csv(csvs[0])
df.columns = df.columns.str.strip()
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED)
print(f'Shape: {df.shape}')
print(df.columns.tolist())
df.head()

Files: ['C:\\Users\\ahmad\\.cache\\kagglehub\\datasets\\adityaramachandran27\\world-air-quality-index-by-city-and-coordinates\\versions\\1\\AQI and Lat Long of Countries.csv']
Shape: (16695, 14)
['Country', 'City', 'AQI Value', 'AQI Category', 'CO AQI Value', 'CO AQI Category', 'Ozone AQI Value', 'Ozone AQI Category', 'NO2 AQI Value', 'NO2 AQI Category', 'PM2.5 AQI Value', 'PM2.5 AQI Category', 'lat', 'lng']


,Country,City,AQI Value,AQI Category,CO AQI Value,CO AQI Category,Ozone AQI Value,Ozone AQI Category,NO2 AQI Value,NO2 AQI Category,PM2.5 AQI Value,PM2.5 AQI Category,lat,lng
0,Russian Federation,Praskoveya,51,Moderate,1,Good,36,Good,0,Good,51,Moderate,44.7444,44.2031
1,Brazil,Presidente Dutra,41,Good,1,Good,5,Good,1,Good,41,Good,-5.2900,-44.4900
2,Brazil,Presidente Dutra,41,Good,1,Good,5,Good,1,Good,41,Good,-11.2958,-41.9869
3,Italy,Priolo Gargallo,66,Moderate,1,Good,39,Good,2,Good,66,Moderate,37.1667,15.1833
4,Poland,Przasnysz,34,Good,1,Good,34,Good,0,Good,20,Good,53.0167,20.8833


## Data Validation

In [5]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

# Find target — look for AQI Category or similar
cat_cols = [c for c in df.columns if 'category' in c.lower() or 'aqi' in c.lower()]
print(f'\nCandidate target cols: {cat_cols}')

# Use AQI Category if exists, else create from AQI Value
if any('category' in c.lower() for c in cat_cols):
    TARGET = [c for c in cat_cols if 'category' in c.lower()][0]
elif any('aqi' in c.lower() for c in df.columns):
    aqi_col = [c for c in df.columns if 'aqi' in c.lower() and 'category' not in c.lower()][0]
    # Create categories from AQI value
    df[aqi_col] = pd.to_numeric(df[aqi_col], errors='coerce')
    df = df.dropna(subset=[aqi_col])
    bins = [0, 50, 100, 150, 200, 300, 500]
    labels = ['Good', 'Moderate', 'Unhealthy_Sensitive', 'Unhealthy', 'Very_Unhealthy', 'Hazardous']
    df['AQI_Category'] = pd.cut(df[aqi_col], bins=bins, labels=labels, include_lowest=True)
    df = df.dropna(subset=['AQI_Category'])
    TARGET = 'AQI_Category'
else:
    TARGET = df.columns[-1]

print(f'\nTarget: {TARGET}')
print(df[TARGET].value_counts())

Missing values:
Country               302
City                    0
AQI Value               0
AQI Category            0
CO AQI Value            0
CO AQI Category         0
Ozone AQI Value         0
Ozone AQI Category      0
NO2 AQI Value           0
NO2 AQI Category        0
PM2.5 AQI Value         0
PM2.5 AQI Category      0
lat                     0
lng                     0
dtype: int64

Duplicates: 0

Candidate target cols: ['AQI Value', 'AQI Category', 'CO AQI Value', 'CO AQI Category', 'Ozone AQI Value', 'Ozone AQI Category', 'NO2 AQI Value', 'NO2 AQI Category', 'PM2.5 AQI Value', 'PM2.5 AQI Category']

Target: AQI Category
AQI Category
Good                              7708
Moderate                          7054
Unhealthy                          871
Unhealthy for Sensitive Groups     869
Very Unhealthy                     131
Hazardous                           62
Name: count, dtype: int64


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('AQI Category Distribution')
axes[0,0].tick_params(axis='x', rotation=45)

num_cols = df.select_dtypes('number').columns.tolist()[:3]
for i, col in enumerate(num_cols):
    ax = axes[(i+1)//2, (i+1)%2]
    ax.hist(df[col].dropna(), bins=40, edgecolor='black')
    ax.set_title(col)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].value_counts())
print(f'\nNumber of classes: {df[TARGET].nunique()}')

AQI Category
Good                              7708
Moderate                          7054
Unhealthy                          871
Unhealthy for Sensitive Groups     869
Very Unhealthy                     131
Hazardous                           62
Name: count, dtype: int64

Number of classes: 6


## Train/Test Split

In [8]:
# Drop non-feature text columns
drop_cols = [c for c in df.columns if any(kw in c.lower() for kw in ['city','country','name','lat','lng','coord'])]
df = df.drop(columns=[c for c in drop_cols if c in df.columns and c != TARGET], errors='ignore')

# Encode target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df[TARGET] = le.fit_transform(df[TARGET].astype(str))
print(f'Classes: {le.classes_}')

# Encode remaining categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    df[c] = df[c].astype('category').cat.codes

df = df.fillna(df.median(numeric_only=True))
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Classes: ['Good' 'Hazardous' 'Moderate' 'Unhealthy'
 'Unhealthy for Sensitive Groups' 'Very Unhealthy']
Train: (13356, 9), Test: (3339, 9)


## Preprocessing
Dropped location text columns. Encoded categoricals. Imputed missing with median.

## Feature Engineering
Using pollutant concentrations directly. Tree models capture non-linear AQI thresholds.

## Baseline Model

In [9]:
bl = LogisticRegression(max_iter=1000, random_state=SEED)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred, average="weighted"):.4f}')

Baseline LR: Acc=0.9913  F1=0.9906


## LazyPredict Benchmark

In [10]:
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  \
Model                                                                         
BaggingClassifier              1.000           1.000000  1.000000  1.000000   
DecisionTreeClassifier         1.000           1.000000  1.000000  1.000000   
XGBClassifier                  1.000           1.000000  1.000000  1.000000   
LGBMClassifier                 1.000           1.000000  1.000000  1.000000   
CatBoostClassifier             1.000           1.000000  1.000000  1.000000   
ExtraTreesClassifier           0.999           0.958333  1.000000  0.998967   
LabelPropagation               0.999           0.958333  0.999999  0.998967   
RandomForestClassifier         0.999           0.958333  1.000000  0.998967   
LabelSpreading                 0.999           0.958333  0.999999  0.998967   
ExtraTreeClassifier            0.997           0.927731  0.998449  0.996946   
SVC                            0.997           0.913

## FLAML AutoML

In [11]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Accuracy={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
models = {}
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

xgb = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p, average='weighted'):.4f}")

CatBoost: Acc=0.9991  F1=0.9991
LightGBM: Acc=1.0000  F1=1.0000
XGBoost: Acc=1.0000  F1=1.0000


## Top 3 Model Selection

In [13]:
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

          Accuracy        F1  Precision    Recall
LightGBM  1.000000  1.000000   1.000000  1.000000
XGBoost   1.000000  1.000000   1.000000  1.000000
CatBoost  0.999102  0.999062   0.999194  0.999102

Top 3: ['LightGBM', 'XGBoost', 'CatBoost']


## Final Evaluation of Top 3

In [14]:
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
    print(f"{name}: Acc={final_results[name]['Accuracy']:.4f}  F1={final_results[name]['F1']:.4f}")
    print(classification_report(y_test, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

LightGBM: Acc=1.0000  F1=1.0000


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1542
           1       1.00      1.00      1.00        12
           2       1.00      1.00      1.00      1411
           3       1.00      1.00      1.00       174
           4       1.00      1.00      1.00       174
           5       1.00      1.00      1.00        26

    accuracy                           1.00      3339
   macro avg       1.00      1.00      1.00      3339
weighted avg       1.00      1.00      1.00      3339

XGBoost: Acc=1.0000  F1=1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1542
           1       1.00      1.00      1.00        12
           2       1.00      1.00      1.00      1411
           3       1.00      1.00      1.00       174
           4       1.00      1.00      1.00       174
           5       1.00      1.00      1.00        26

    accuracy                           1.00  

## Error Analysis

In [15]:
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    axes[1].text(0.5, 0.5, 'No feature importance', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

wrong = X_test[preds != y_test]
print(f'Correct: {(preds == y_test).sum()}, Wrong: {len(wrong)}, Error rate: {len(wrong)/len(y_test):.4f}')

Correct: 3339, Wrong: 0, Error rate: 0.0000


## Interpretation & Business Insight
PM2.5 and PM10 are the most influential pollutants for AQI classification. Models can help cities prioritize air quality interventions.

## Limitations
- AQI categories are deterministic from values — ML is overkill if exact formulas known
- Missing data in some pollutant readings
- Geographic bias in monitoring coverage

## How to Improve
- Add weather features (wind, humidity)
- Time series for forecasting
- Satellite-derived features for unmonitored areas

## Production Considerations
- Real-time pollutant feed integration
- Health advisory triggers
- Spatial interpolation for coverage gaps

## Common Mistakes
- Including AQI value as feature (leakage — it determines category)
- Ignoring class imbalance across categories
- Not validating against known AQI formulas

## Mini Challenge
1. Remove the AQI value and predict only from raw pollutants
2. Binary: healthy vs unhealthy
3. Add city population as feature

## Final Summary
Classified air quality categories from pollutant data. Boosting models handle the multi-class task effectively.

In [16]:
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "LightGBM": {
    "Accuracy": 1.0,
    "F1": 1.0,
    "Precision": 1.0,
    "Recall": 1.0
  },
  "XGBoost": {
    "Accuracy": 1.0,
    "F1": 1.0,
    "Precision": 1.0,
    "Recall": 1.0
  },
  "CatBoost": {
    "Accuracy": 0.9991,
    "F1": 0.9991,
    "Precision": 0.9992,
    "Recall": 0.9991
  },
  "best_model": "LightGBM"
}
